# Hybrid Topic: fit, inspect, save, and reuse

This tutorial is for Python users who want to inspect topic assignments in Jupyter.
You will fit a small model, inspect its results, export tables, and reuse a saved model.

The executable cells use **fixed example topics and toy word features**. They need no
API key, model download, or project dataset. This demonstrates the software workflow;
it does not demonstrate automatic topic discovery or measure semantic quality.
The final section shows how to configure real discovery for your own texts.

From the repository root, install and launch:

```bash
python -m pip install -e '.[notebook]'
python -m jupyterlab examples/quickstart.ipynb
```

Select the Python environment where you installed Hybrid Topic. Use **Restart Kernel
and Run All Cells** to run the complete offline tutorial. Python 3.12 on macOS is the
locally checked environment; other platforms have not yet been verified.

## 1. Prepare a small text collection

Each string is one document. Input order and duplicate texts are preserved; empty
strings are rejected. At least two documents are required for fitting.

In [ ]:
from pathlib import Path
from tempfile import mkdtemp

import numpy as np
import pandas as pd

from hybrid_topic import HybridTopic, read_documents
from hybrid_topic import Topic

# Keep each run separate so rerunning the notebook does not overwrite prior work.
output_root = Path("hybrid_topic_outputs")
output_root.mkdir(exist_ok=True)
run_dir = Path(mkdtemp(prefix="quickstart-", dir=output_root)).resolve()

documents = [
    "The battery is short lived",
    "The battery needs a charge",
    "Delivery was delayed",
    "Shipping and delivery were fast",
    "Unrelated finance comment",
]
pd.DataFrame({"text": documents})

## 2. Fit a model with existing topics

The small encoder below recognizes only the listed English words. It is a teaching
fixture, so it will reject text with none of those words. For arbitrary text, replace
it with a real embedding model, as shown at the end.

The fixed starting profile is `p=0.95`, `seed_per_topic=5`,
`diffusion_rate=0.70`, `rounds=10`, and `max_residual_rounds=2`. You can omit these
arguments or override them explicitly. This profile has passed a limited
engineering check; it is not universally optimal and assignments can change when
parameters change. Scores are not calibrated probabilities.

In [ ]:
class DemoEmbedder:
    model_name = "hybrid-topic-offline-demo-v1"

    def encode(self, texts):
        groups = [("battery", "charge"), ("delivery", "shipping"), ("unrelated", "finance")]
        return np.asarray([
            [float(any(term in text.casefold() for term in group)) for group in groups]
            for text in texts
        ])


codebook = [
    Topic("T1", "Battery", "Battery and charging experience", ("battery", "charge")),
    Topic("T2", "Delivery", "Delivery and shipping experience", ("delivery", "shipping")),
]
embedder = DemoEmbedder()
model = HybridTopic(embedding_model=embedder)
topic_ids, scores = model.fit_transform(documents, codebook=codebook)
model.get_topic_info()

The topic table contains the model's topic IDs, original source IDs, names,
definitions, keywords, and document counts. IDs remain stable within this saved model.
For this fixture, Battery and Delivery each contain two training documents.

In [ ]:
document_info = model.get_document_info()
assert topic_ids.tolist() == [0, 0, 1, 1, -1]
assert model.get_topic_info()["count"].tolist() == [2, 2]
document_info

## 3. Inspect a topic and unassigned documents

`-1` means unassigned. Assignment scores are **not probabilities**. An unassigned
text should remain visible for inspection; it is not silently forced into a topic.

In [ ]:
model.get_document_info(topic_id=0)

In [ ]:
model.get_document_info(topic_id=-1)

## 4. Export results and read a CSV text column

Exports contain `topics.csv`, `documents.csv`, and `result.json`. Existing result
files are protected against overwriting. Keep exported tables when you want the
original text later: model saves do not include raw training documents.

The CSV loader reads only the named text column and does not use labels or other
columns to discover topics. Here we create a small example CSV using the same texts.

In [ ]:
export_dir = model.export(run_dir / "training_results")
source_csv = run_dir / "example_comments.csv"
pd.DataFrame({"body": documents, "source": "tutorial"}).to_csv(source_csv, index=False)
loaded_documents = read_documents(source_csv, text_column="body")
assert loaded_documents == documents
print("Exported:", ", ".join(sorted(p.name for p in export_dir.iterdir())))
print("Output directory:", run_dir)

## 5. Save, reload, and assign new texts

A model save contains the topics, fixed training references, scores, threshold, and
configuration. Supply the same embedding model and preprocessing when loading.
It can show topic summaries and assign new text; training text remains in your
separate result export.

New texts are encoded one at a time and query fixed training references. They do
not influence one another, change the threshold, or add topics. Use a deterministic
embedding backend. Training and new-text inference are different operations:
reprocessing a training text with `transform` need not reproduce its fit assignment.

In [ ]:
model.save(run_dir / "model")
restored = HybridTopic.load(run_dir / "model", embedding_model=DemoEmbedder())
new_documents = ["Battery charge issue", "Delivery arrived", "Unrelated finance text"]
new_result = restored.transform_result(new_documents)
new_result.export(run_dir / "new_document_results")
assert new_result.get_document_info()["topic_id"].tolist() == [0, 1, -1]
new_result.get_document_info()

## 6. Try changing the query batch

Exercise: move the battery comment to a different position and include it twice.
Its assignment and score should match those obtained when it is processed alone.
The next cell checks this expectation. You can add another delivery or finance text.

In [ ]:
alone_ids, alone_scores = restored.transform(["Battery charge issue"])
mixed = ["Delivery arrived", "Battery charge issue", "Unrelated finance text", "Battery charge issue"]
mixed_ids, mixed_scores = restored.transform(mixed)
np.testing.assert_array_equal(mixed_ids[[1, 3]], np.repeat(alone_ids, 2))
np.testing.assert_array_equal(mixed_scores[[1, 3]], np.repeat(alone_scores, 2))
print("The same text keeps the same topic and score across batches.")

## 7. Use your own texts and discover topics

For real discovery, install the embedding and OpenAI extras from the repository root:

```bash
python -m pip install -e '.[embeddings,openai,notebook]'
```

Provide `OPENAI_API_KEY`, `OPENAI_MODEL`, and `MODEL_CONTEXT_WINDOW` through your environment. Do not put
credentials in notebook cells or exported outputs. The following is an **optional,
unexecuted example**: running it may download embedding model weights and sends a
packed sample of your input texts to the configured API, which can incur charges.

```python
import os
from hybrid_topic.backends import OpenAITopicGenerator, SentenceTransformerEmbedder

texts = read_documents("my_comments.csv", text_column="text")
real_embedder = SentenceTransformerEmbedder(
    "Qwen/Qwen3-Embedding-0.6B", device="cpu", local_files_only=False,
)
generator = OpenAITopicGenerator(
    model=os.environ["OPENAI_MODEL"],
    api_key=os.environ["OPENAI_API_KEY"],
    context_window=int(os.environ["MODEL_CONTEXT_WINDOW"]),
    # Optional: omit for general topic discovery.
    topic_instruction="Identify recurring product issues",
)
real_model = HybridTopic(generator=generator, embedding_model=real_embedder)
real_model.fit(texts)  # No supplied codebook: the generator discovers initial topics.
real_model.get_topic_info()
real_model.get_document_info()
```

The same interface accepts news, comments, and other text collections. General
purpose does not mean quality has already been validated for every language or domain.
The embedding adapter reads up to 512 tokens per text. Generation uses whole documents
within 75% of the configured context, reserving output space (default 16,384 tokens).
It samples reproducibly only when all candidate documents cannot fit. Long-text quality
needs separate validation. Fit builds a dense document graph, so large collections require
memory checks. Generator-backed fit now adds up to two residual discovery rounds by default,
with early stopping when no valid new topics remain. Set `max_residual_rounds=0`
to disable this or `1` for one extra round. Supplied codebooks, including this
offline tutorial, skip all generation. Inspect `real_model.metadata_` for round
history and generator call counts. Format repair defaults to one additional attempt;
set `max_format_retries=0` to disable it. SDK transport retries are disabled.
The default percentile is `p=0.95`; pass `p=...` to override it.
`metadata_["configuration"]` records the actual settings, and
`metadata_["parameter_defaults_version"]` identifies the library reference profile.
Independent semantic quality evaluation remains necessary.

If an import fails, install the named extra into the notebook's Python environment
and restart the kernel. If a real embedding model is unavailable locally, either
allow its download explicitly or choose a model you already have. The toy encoder
above is only for this fixed offline fixture.
The same cloud workflow is available as a runnable script from the checkout:

```bash
python -m examples.cloud_workflow --output my_cloud_run --allow-download
```

Omit `--allow-download` to require cached weights. Use `--csv comments.csv
--text-column text` for your own data and `--direction "Identify recurring product issues"`
for optional guidance. Choose a new output directory for each run.
